In [1]:
import subprocess
import tempfile
from itertools import chain
from pathlib import Path

from IPython.display import Markdown, display
from tqdm.notebook import tqdm

import ghapicache
import githelpers
import release_notes

ReleaseMetadata = release_notes.ReleaseMetadata

In [2]:
%load_ext autoreload
%autoreload 1
%aimport githelpers
%aimport ghapicache
%aimport release_notes

In [3]:
local_repo = Path.home() / "Code/celeritas"
githelpers.REPO = local_repo

# TODO: implement stashed code
# git = githelpers.GitRunner(local_repo)
# release_notes.GIT = git

In [4]:
cached = ghapicache.GhApiCache(repo="celeritas")
ghapi = cached.api
# cached.purge()

In [5]:
int(ghapi.limit_rem)

5000

# Load all prs

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# PR parser uses local data
pr_parser = release_notes.PullRequestLogParser(Path("data/github-sha-map.json"))


def parallel_load_contributions(cached: release_notes.GhApiCache, pull_ids: list[int]):
    contributions = []
    executor = ThreadPoolExecutor(max_workers=8)
    futures = []

    try:
        futures = [
            executor.submit(release_notes.load_contributions, cached, pr)
            for pr in pull_ids
        ]
        for future in tqdm(as_completed(futures), total=len(futures)):
            contributions.append(future.result())
    except KeyboardInterrupt:
        for future in futures:
            future.cancel()
        executor.shutdown(wait=False, cancel_futures=True)
        print("Shut down early: contributions are incomplete")
    else:
        executor.shutdown(wait=True)

    return contributions


# Convenience function uses local cache
def load_contributions(
    release_md: release_notes.ReleaseMetadata,
):
    prs = release_notes.PullRequestRange(release_md, pr_parser)

    contributions = []
    for pr_id in tqdm(prs.pull_ids):
        contributions.append(release_notes.load_contributions(cached, pr_id))

    return release_notes.merge_contributions(contributions)


# All commits from all time
develop_sha = githelpers.git_rev_parse("upstream/develop")
all_md = ReleaseMetadata(
    release=f"develop-{develop_sha}",
    merge_bases=["v0.0.0"],
    target_branch=develop_sha,
)
all_prs = release_notes.PullRequestRange(all_md, pr_parser)
contributions = parallel_load_contributions(cached, all_prs.pull_ids)
all_contrib = release_notes.merge_contributions(contributions)
cached.flush()

Ignore 'Format code base (clang-format version 11.0.1)' (7093314f26c023d664e1ff65a154fd5b4c699c03): old clang-format
Ignore 'Fix linking to CUDA toolkit when using VecGeom' (34e8a82fee4f6a7ede5d8c574aaa57c9acdf0a6d): hotfix vecgeom
Ignore 'Add option to use spline-interpolated energy loss (#1496)' (d2871473f447f27b97b43876258f87bb17c08746): weird github pull glitch from 'new feature'


  0%|          | 0/2092 [00:00<?, ?it/s]

Saved cache to: data/ghapicache-celeritas-project-celeritas.json


# Load teams and process members

In [10]:
# Load teams
TEAMS = {}
for t in tqdm(cached.teams()):
    TEAMS[t["name"]] = release_notes.get_team(cached, t)

team_members = frozenset(chain(*(t.members for t in TEAMS.values())))
org_members = frozenset(m["login"] for m in cached.org_members())
TEAMS["affiliates"] = release_notes.Team(
    description="Associated with Celeritas but not core members",
    members=(org_members - team_members),
)

# Initialize the user cache
user_cache = release_notes.UserCache(
    cached, local_repo / "scripts/release/contributors.jsonl"
)
for login in tqdm(org_members):
    user_cache[login]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

In [50]:
!open {user_cache.json_path}

In [11]:
user_cache.flush()

## List active members

This is to be used for crediting in presentations, etc.

In [52]:
text = []
for team, title in [
    ("code-lead", "Code lead"),
    ("core", "Core members"),
    ("core-advisor", "Core advisors"),
    ("affiliates", "Affiliates"),
]:
    text.append(f"## {title}")
    m = [user_cache[username] for username in TEAMS[team].members]
    for member in sorted(m, key=release_notes.get_last_name):
        if member.name is None:
            print(f"Missing metadata: {member!r}")
            continue
        text.append("- " + release_notes.format_user(member))
    text.append("")

display(Markdown("\n".join(text)))

Missing metadata: UserInfo(name=None, institute=None, email=None, orcid=None, github='erica-snider')


## Code lead
- Seth R. Johnson *(ORNL)*

## Core members
- Elliott Biondo *(ORNL)*
- Greg Davidson *(ORNL)*
- Julien Esseiva *(LBNL)*
- Rashika Gupta *(ORNL)*
- Hayden Hollenbeck *(UVA)*
- Seth R. Johnson *(ORNL)*
- Soon Yung Jun *(FNAL)*
- Guilherme Lima *(FNAL)*
- Amanda Lund *(ANL)*
- Ben Morgan *(U Warwick)*
- Sakib Rahman *(BNL)*
- Loren Schwiebert *(Wayne State U)*
- Stefano Tognini *(ORNL)*

## Core advisors
- Philippe Canal *(FNAL)*
- Marcel Demarteau *(ORNL)*
- Tom Evans *(ORNL)*

## Affiliates
- Lance Bullerwell *(ORNL)*
- Wouter Deconinck *(U Manitoba)*
- Sam Eriksen *(U Bristol)*
- Sanmay Ganguly *(None)*
- Steven Hamilton *(ORNL)*
- Aaron Higuera *(None)*
- Ilker Parmaksiz *(None)*
- Kevin Pedro *(None)*
- Paul Romano *(ANL)*
- Frederic Suter *(ORNL)*
- Sandro Wenzel *(CERN)*


In [53]:
collaborators = [user_cache[username] for username in user_cache.users]

In [54]:
collaborators[0]._asdict()

{'name': 'Aaron Higuera',
 'institute': None,
 'email': None,
 'orcid': None,
 'github': 'ahiguera-mx'}

In [55]:
import csv

# Write collaborators to CSV
with open("collaborators.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["last", "first", "orcid", "institution"])

    for collab in sorted(collaborators, key=release_notes.get_last_name):
        # Split name into first and last
        if collab.name:
            name_parts = collab.name.strip().split()
            if len(name_parts) > 1:
                last = name_parts[-1]
                first = " ".join(name_parts[:-1])
            else:
                last = name_parts[0]
                first = ""
        else:
            first = ""
            last = ""

        writer.writerow([last, first, collab.orcid or "", collab.institute or ""])

print(f"Wrote {len(collaborators)} collaborators to collaborators.csv")

Wrote 39 collaborators to collaborators.csv


## CHEP authors and contributors

In [12]:
for username in tqdm(set(all_contrib.author) - set(user_cache.users)):
    print(user_cache[username])

0it [00:00, ?it/s]

In [13]:
INST_ADDRESS = {
    "CERN": "CERN, Geneva, Switzerland",
    "ORNL": "Oak Ridge National Laboratory, Oak Ridge, TN, USA",
    "U Warwick": "Department of Physics, University of Warwick, Coventry, United Kingdom",
    "FNAL": "Fermi National Accelerator Laboratory, Batavia, IL, USA",
    "ANL": "Argonne National Laboratory, Lemont, IL, USA",
    "LBNL": "Lawrence Berkeley National Laboratory, Berkeley, CA, USA",
    "UVA": "University of Virginia, Charlottesville, VA, USA",
    "Wayne State U": "Wayne State University, Detroit, MI, USA",
    "BNL": "Brookhaven National Laboratory, Upton, NY, USA",
}


def ordered_uniquify(x: list):
    result: list = []
    seen = set()

    for v in x:
        if v in seen:
            continue
        result.append(v)
        seen.add(v)

    return result


def print_chep_authors(contributors, institutes):
    """Print CHEP author blocks given contributors and ordered institute list."""
    institute_map = {inst: i for i, inst in enumerate(institutes, start=1)}
    author_blocks = []

    for c in contributors:
        parts = c.name.split()
        if len(parts) > 1:
            first = " ".join(parts[:-1])
            last = parts[-1]
        else:
            first = ""
            last = parts[0]

        inst_index = institute_map[c.institute]
        inst_suffix = f"\\inst{{{inst_index}}}"
        orcid_suffix = f"\n  \\orcidlink{{{c.orcid}}}%" if c.orcid else ""

        author_blocks.append(
            f"  \\firstname{{{first}}} \\lastname{{{last}}}{inst_suffix}%{orcid_suffix}"
        )

    if not author_blocks:
        print("  No authors found")
        return

    print("\n  %%\n  \\and\n".join(author_blocks))


def print_chep_institutes(institutes):
    """Print institute addresses joined with \and."""
    addresses = []
    for institute in institutes:
        addresses.append(INST_ADDRESS[institute])

    if not addresses:
        print("  No institutes found")
        return

    print("\n  \\and\n".join(f"  {addr}" for addr in addresses))

In [14]:
chep: list[release_notes.UserInfo] = []
for team in ["code-lead", "core", "core-advisor"]:
    users = [user_cache[username] for username in TEAMS[team].members]
    users.sort(key=lambda u: u.name.split()[-1])
    for u in users:
        if u.institute is None:
            print(f"missing insitute: {u}")
            continue
        chep.append(u)

chep

print({u.institute for u in chep} - set(INST_ADDRESS))

set()


In [15]:
institutes = ordered_uniquify(["CERN", "ORNL"] + [u.institute for u in chep])
print_chep_authors(chep, institutes)
print_chep_institutes(institutes)

  \firstname{Seth R.} \lastname{Johnson}\inst{2}%
  \orcidlink{0000-0003-1504-8966}%
  %%
  \and
  \firstname{Elliott} \lastname{Biondo}\inst{2}%
  \orcidlink{0000-0002-9088-1360}%
  %%
  \and
  \firstname{Greg} \lastname{Davidson}\inst{2}%
  \orcidlink{0000-0002-3941-8688}%
  %%
  \and
  \firstname{Julien} \lastname{Esseiva}\inst{3}%
  \orcidlink{0009-0002-1119-4851}%
  %%
  \and
  \firstname{Rashika} \lastname{Gupta}\inst{2}%
  \orcidlink{0009-0003-0658-9549}%
  %%
  \and
  \firstname{Hayden} \lastname{Hollenbeck}\inst{4}%
  \orcidlink{0009-0000-6359-3294}%
  %%
  \and
  \firstname{Seth R.} \lastname{Johnson}\inst{2}%
  \orcidlink{0000-0003-1504-8966}%
  %%
  \and
  \firstname{Soon Yung} \lastname{Jun}\inst{5}%
  \orcidlink{0000-0003-3370-6109}%
  %%
  \and
  \firstname{Guilherme} \lastname{Lima}\inst{5}%
  \orcidlink{0000-0003-4585-0546}%
  %%
  \and
  \firstname{Amanda} \lastname{Lund}\inst{6}%
  \orcidlink{0000-0002-8316-0709}%
  %%
  \and
  \firstname{Ben} \lastname{Morgan}\inst{

## Print authors since the last major release for presentations

In [ ]:
latest_major_md = ReleaseMetadata(
    release="recent-develop",
    merge_bases=["v0.7.0-dev"],
    target_branch="upstream/develop",
)

prs = release_notes.PullRequestRange(latest_major_md, pr_parser)
contributions = []
for pr_id in tqdm(prs.pull_ids):
    contributions.append(release_notes.load_contributions(cached, pr_id))
merged_contrib = release_notes.merge_contributions(contributions)

text = ["# Current contributors"]
current_contrib = set()
for username in merged_contrib.author:
    m = user_cache[username]
    text.append(f"- {m.name} (@{m.github})")
    current_contrib.add(m.github)
text.extend(["", "# Past contributors"])
for username in all_contrib.author:
    if username not in current_contrib:
        m = user_cache[username]
        text.append(f"- {m.name} (@{username})")
# display(Markdown("\n".join(text)))
print("\n".join(text))

Ignore 'Update cmake presets' (b427d6915c82e20f2c1312b9d647ece6fce3d6ae): presets on backport branch
Ignore 'Write setup Geant4 log messages to world logger, not self logger' (cc6f24d1716c9e2f652f9db3e732cf82f056c76e): backported component from #1895


  0%|          | 0/624 [00:00<?, ?it/s]

# Current contributors
- Seth R. Johnson (@sethrj)
- Amanda Lund (@amandalund)
- Julien Esseiva (@esseivaju)
- Elliott Biondo (@elliottbiondo)
- Hayden Hollenbeck (@hhollenb)
- Rashika Gupta (@Rashika-Gupta)
- Philippe Canal (@pcanal)
- Stefano Tognini (@stognini)
- Loren Schwiebert (@LSchwiebert)
- Soon Yung Jun (@whokion)
- Greg Davidson (@davidsgr)
- Sakib Rahman (@rahmans1)
- Owen Strong (@osanstrong)
- Lance Bullerwell (@lebuller)
- Guilherme Lima (@mrguilima)
- Ben Morgan (@drbenmorgan)
- Tarek Gaddar (@tghaddar)
- Stephen Nicholas Swatman (@stephenswat)

# Past contributors
- Paul Romano (@paulromano)
- Vincent R. Pascuzzi (@vrpascuzzi)
- Tom Evans (@tmdelellis)
- Vidor Heli Lujan Montiel (@VHLM2001)
- Damien L-G (@dalg24)
- Doaa Deeb (@DoaaDeeb)
- Peter Heywood (@ptheywood)
- Andrey Prokopenko (@aprokop)
- Shane Hart (@hartsw)


In [ ]:
display(Markdown("\n".join(text)))

# Current contributors
- Seth R. Johnson (@sethrj)
- Amanda Lund (@amandalund)
- Julien Esseiva (@esseivaju)
- Elliott Biondo (@elliottbiondo)
- Hayden Hollenbeck (@hhollenb)
- Rashika Gupta (@Rashika-Gupta)
- Philippe Canal (@pcanal)
- Stefano Tognini (@stognini)
- Loren Schwiebert (@LSchwiebert)
- Soon Yung Jun (@whokion)
- Greg Davidson (@davidsgr)
- Sakib Rahman (@rahmans1)
- Owen Strong (@osanstrong)
- Lance Bullerwell (@lebuller)
- Guilherme Lima (@mrguilima)
- Ben Morgan (@drbenmorgan)
- Tarek Gaddar (@tghaddar)
- Stephen Nicholas Swatman (@stephenswat)

# Past contributors
- Paul Romano (@paulromano)
- Vincent R. Pascuzzi (@vrpascuzzi)
- Tom Evans (@tmdelellis)
- Vidor Heli Lujan Montiel (@VHLM2001)
- Damien L-G (@dalg24)
- Doaa Deeb (@DoaaDeeb)
- Peter Heywood (@ptheywood)
- Andrey Prokopenko (@aprokop)
- Shane Hart (@hartsw)

In [ ]:
cached.flush()

# Release process

- Merge base should be all commits *already* released (skip documenting)
- Target branch is the one where the release candidate is
- Previous major branch allows all "v.x" contributors to be credited

In [16]:
# Major release
major_md = release_notes.ReleaseMetadata(
    release="0.7.0", merge_bases=["v0.6.0", "v0.6.3"], target_branch="upstream/develop"
)

# Minor release
minor_md = release_notes.ReleaseMetadata(
    release="0.6.3", merge_bases=["v0.6.2"], target_branch="backports/v0.6"
)

# Release candidate (atypical)
rc_md = release_notes.ReleaseMetadata(
    release="0.7.0-rc.1",
    # merge_bases=["v0.6.0", "v0.6.3"],
    merge_bases=["v0.7.0-dev^"],
    target_branch="upstream/develop",
)

## Release note generation

In [17]:
if 0:
    # Backport release:
    release_md = minor_md
    note_body = """
Version {release} is a minor bug fix update.

- Support CUDA 13
- Allow initialization of Geant4 integration before run manager
"""
elif False:
    # Major release
    release_md = major_md
    note_body = """
Version {release} is a major update to Celeritas featuring many new capabilities.

-

A few minor features are noteworthy:

- 

Notable bug fixes include:

- 

Some interfaces have been removed:

- 

Known deficiencies:

- 
"""

else:
    # 0.7
    release_md = major_md
    note_body = """
Version {release} is a long-overdue major update to Celeritas featuring many new capabilities.

- Optical physics is now fully supported, including surface properties, Geant4
  process integration, and Mie/WLS processes.
- A LArSoft plugin that stands in for PDFastSimPar uses full-fidelity MC transport
  on CPU or GPU for optical photon transport from simulated energy deposition.
- The EM tracking loop is now asynchronous (relying on built-in CUDA asynchronous
  streaming and events rather than an additional control thread).
- ORANGE geometry navigation is now substantially faster and now
  completely supports Geant4 quadric solids.
- G4 tracking manager integration is now set up automatically rather than
  requiring BeginOfRun/EndOfRun actions.

Major known deficiencies include:

- Surface normal calculations for VecGeom is unsupported, so optical physics
  will be wrong if any surface physics is used.
- The behavior of reflectivity/efficiency grids acting as pure absorbers
  around sensitive detector regions (i.e., black boxes for splicing)
  currently differs from Geant4.
- Tessellated solids are not supported when using ORANGE.
"""

In [18]:
prs = release_notes.PullRequestRange(release_md, pr_parser)
sorted_pulls = release_notes.SortedPulls(cached)

contributions = []
for pr_id in tqdm(prs.pull_ids):
    try:
        # del cached.cache["pull"][cached.subkey(pr_id)]
        sorted_pulls.add(pr_id)
        contributions.append(release_notes.load_contributions(cached, pr_id))
    except Exception as e:
        print(f"Error adding PR #{pr_id}: {e}")
        del cached.cache["pull"][cached.subkey(pr_id)]

reviewers = release_notes.merge_contributions(contributions).reviewer
for login in tqdm(reviewers):
    user_cache[login]

Ignore 'Update cmake presets' (b427d6915c82e20f2c1312b9d647ece6fce3d6ae): presets on backport branch
Ignore 'Write setup Geant4 log messages to world logger, not self logger' (cc6f24d1716c9e2f652f9db3e732cf82f056c76e): backported component from #1895


  0%|          | 0/624 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

In [19]:
cached.flush()

In [20]:
def make_notes(cls, release_md, note_body):
    prev = release_md.merge_bases[0]
    other = release_md.merge_bases[1:]
    if not other:
        change_str = f"Changes since {prev} follow."
    else:
        # NOTE: this is redundant if prev is a merge base of other
        assert len(other) == 1
        change_str = (
            f"Changes since {prev}, excluding those released in {other[0]}, follow."
        )

    notes = cls(release_md, note_body)
    notes.paragraph(change_str)
    notes.sorted_pulls(sorted_pulls)
    notes.reviewers(reviewers, user_cache)
    notes.changelog_line("celeritas-project", "celeritas")

    return notes

In [21]:
rst_notes = make_notes(release_notes.RstNotes, release_md, note_body)

with githelpers.open_pbcopy() as pb:
    rst_notes.write(pb)

print("Release notes copied to clipboard")
subprocess.run(["open", str(githelpers.REPO / "doc/release-history")])

Release notes copied to clipboard


CompletedProcess(args=['open', '/Users/seth/Code/celeritas/doc/release-history'], returncode=0)

## Create github release

Make sure the release history above is *first* merged into upstream.
For backport releases, *also* ensure it is **cherry-picked into the backport branch**.

First, tag the release locally and update the target to the tagged commit.

In [22]:
release_md

ReleaseMetadata(release='0.7.0', merge_bases=['v0.6.0', 'v0.6.3'], target_branch='upstream/develop')

In [27]:
if True:
    print(githelpers.git("remote", "update"))
    subject = githelpers.git("log", "-1", r"--format=%s", release_md.target_branch)[0]
    if not subject.startswith("Release v" + release_md.release):
        print(
            f"WARNING: expected last commit subject to include release, but subject line is '{subject}'"
        )

    release_md = release_notes.create_local_tag(release_md)
else:
    release_md.target_branch = githelpers.gitz(
        "log", "-1", "-z", "--format=%H", f"v{release_md.release}"
    )[0]

print(release_md)

ReleaseMetadata(release='0.7.0', merge_bases=['v0.6.0', 'v0.6.3'], target_branch='aa36bdf262ee11f4c9718f0faf835e57c77e1f2e')


Then create a gzipped in-memory archive of the repository. Decompress and run `cmake -P cmake/CgvFindVersion.cmake` to check the version is exported correctly.

In [33]:
tgz_data = githelpers.git_archive_tgz(release_md.target_branch)
tgz_name = f"{cached.repo}-{release_md.release}.tar.gz"

with tempfile.TemporaryDirectory(delete=False) as tmpdir:
    with open(Path(tmpdir) / tgz_name, "wb") as f:
        f.write(tgz_data)
    subprocess.run(["open", tmpdir])

In [84]:
subprocess.run(
    ["cmake", "-P", "cmake/CgvFindVersion.cmake"],
    cwd=Path(tmpdir) / f"{cached.repo}-{release_md.release}",
)

VERSION="0.7.0"
VERSION_STRING="0.7.0"


CompletedProcess(args=['cmake', '-P', 'cmake/CgvFindVersion.cmake'], returncode=0)

Now generate the PDF documentation and check it.

In [85]:
build_dir = local_repo / "build"


def run_build(*args):
    return subprocess.run(args, cwd=build_dir, check=True)


githelpers.git("checkout", "--detach", release_md.target_branch)
run_build("cmake", "-UCeleritas_GIT_DESCRIBE", ".")
run_build("ninja", "doc_pdf")
pdf_doc = build_dir / "doc" / "latex" / "Celeritas.pdf"
assert pdf_doc.is_file()
!open {pdf_doc}

-- Configuring done (0.7s)
-- Generating done (0.3s)
-- Build files have been written to: /Users/seth/Code/celeritas/build
[1/14] Creating directories for 'ornltm'
[2/14] Generating Doxygen: doc/doxygen-xml/index.xml
[3/14] Processing Sphinx documentation
[4/14] Performing download step (git clone) for 'ornltm'
Cloning into 'ornltm'...
fatal: unable to access 'https://code.ornl.gov/sethrj/ornltm.git/': The requested URL returned error: 504
Cloning into 'ornltm'...
Had to git clone more than once: 2 times.
HEAD is now at 6379032 Merge branch 'upstream-updates' into 'main'
[5/14] Performing update step for 'ornltm'
-- Already at requested ref: 6379032d2f1e5e8bc1589d087465371fd8b432c1
[6/14] No patch step for 'ornltm'
[7/14] No configure step for 'ornltm'
[8/14] No build step for 'ornltm'
[9/14] No install step for 'ornltm'
[10/14] Completed 'ornltm'
[11/14] Building tech memo footer
Rc files read (in order):
  /Users/seth/.latexmkrc
  ./latexmkrc
Latexmk: This is Latexmk, John Collins, 9

Create the github release draft and upload the tarball and PDF documentation

In [31]:
ghapi = cached.api

markdown_notes = make_notes(release_notes.MarkdownNotes, release_md, note_body)

assert release_md.release is not None
gh_release = release_notes.find_release(ghapi, release_md.release)
if not gh_release:
    print("Creating release")
    gh_release = release_notes.create_release(ghapi, release_md, str(markdown_notes))
    tgz_asset = release_notes.upload_asset(cached, gh_release, tgz_data, tgz_name)
    doc_asset = release_notes.upload_asset(
        cached, gh_release, pdf_doc, f"celeritas-user-manual-{release_md.release}.pdf"
    )
else:
    # Restarting from later:
    # WRONG though
    tgz_data = release_notes.get_tarball(cached, gh_release)

In [34]:
sha256_hash = release_notes.prepare_spack_update("celeritas", release_md, tgz_data)

Spack version copied to clipboard!


In [ ]:
display(
    Markdown(f"""
## Time to publish!

1. Visit [the github release page]({gh_release["html_url"]})
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is {sha256_hash}
4. Double check the name and release notes
5. Edit and publish!
""")
)


## Time to publish

1. Visit [the github release page](https://github.com/celeritas-project/celeritas/releases/tag/untagged-804e7f799a631819ea29)
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is c87910acd0bb9dc85c53e2e4d9958f36d3bb1e5975f0e08fe7274b332a2ccf27
4. Double check the name and release notes
5. Hit publish!


# Push to zenodo

In [49]:
import zenodoapi

%aimport zenodoapi

In [50]:
# Load the Zenodo token
token_path = Path.home() / ".config/zenodo-token"
with open(token_path) as f:
    zenodo_token = f.read().strip()

zenodo = zenodoapi.Zenodo(zenodo_token)
zenodo.api_url = "https://zenodo.org/api/"

In [ ]:
def download_tarball(gh_release):
    asset = gh_release["assets"][0]
    name = asset["name"]
    content = cached.download_file(asset["url"], ext=Path(name).suffix)
    assert content is not None
    return (content, name)


## Minor release

This assumes the major release has already been published upstream into Zenodo.

In [69]:
# assert not release_md.is_major()
major, minor, patch = release_md.as_version()
# release_md = release_notes.ReleaseMetadata.from_comprehensive_version(minor, patch)
major_dep = zenodo.find_deposition(f"Celeritas {major}.{minor}")

In [70]:
zenodo.get_deposition(major_dep.get_latest_version().id)

ZenodoDeposition(id=23104914, title="Celeritas 0.7")

In [71]:
gh_release = release_notes.find_release(ghapi, release_md.release)

In [72]:
make_zenodo_md = release_notes.ZenodoMetadataBuilder(user_cache=user_cache, teams=TEAMS)

In [ ]:
contributions = parallel_load_contributions(cached, all_prs.pull_ids)
all_contrib = release_notes.merge_contributions(contributions)


In [ ]:
# Get the Zenodo metadata
from requests import HTTPError


old_md = major_dep.data["metadata"]
new_md = make_zenodo_md(contributions, release_md, gh_release)
# Only update editors, not team members
old_contrib = new_md["contributors"]
new_contrib = [u for u in old_contrib if u["type"] == "Editor"]
old_contrib = [u for u in old_contrib if u["type"] != "Editor"]
new_md["contributors"] = new_contrib + old_contrib
# Don't change the title
new_md["title"] = major_dep.data["metadata"]["title"]
# Add the body
new_md["description"] = "\n\n".join([
    old_md["description"],
    f"<h1>Version {release_md.release}</h1>",
    new_md["description"],
])
try:
    new_vers = major_dep.create_new_version()
except HTTPError as e:
    # Maybe the version already exists
    if e.response.status_code == 400:
        print("Draft may already exist:", e)
        new_vers = major_dep.get_latest_draft()
    else:
        raise
try:
    new_vers.update(new_md)
except HTTPError as e:
    # Something else went wrong? Try to upload the tarball
    print("Draft may already exist:", e)

# Upload the release
(content, name) = download_tarball(gh_release)
new_vers.upload(content, name)
# The old tarball may still be there (this is buggy) so delete it
new_vers.refresh()
for file in new_vers.get_files():
    if file.filename != name:
        try:
            file.delete()
        except Exception as e:
            print(f"Failed to delete {file.filename}: {e}")
new_vers.refresh()

NameError: name 'load_contributions' is not defined

## Major release

In [36]:
release_md

ReleaseMetadata(release='0.7.0', merge_bases=['v0.6.0', 'v0.6.3'], target_branch='aa36bdf262ee11f4c9718f0faf835e57c77e1f2e')

In [37]:
release_md.as_version()

(0, 7, 0)

In [38]:
assert release_md is not None and release_md.is_major()
major, minor, patch = release_md.as_version()
# release_md = release_notes.ReleaseMetadata.from_comprehensive_version(major, minor)
title = f"Celeritas {major}.{minor}"

In [ ]:
gh_release = release_notes.find_release(ghapi, release_md.release)

In [ ]:
prs = release_notes.PullRequestRange(release_md, pr_parser)

contributions = []
for pr_id in tqdm(prs.pull_ids):
    contributions.append(release_notes.load_contributions(cached, pr_id))

all_contrib = release_notes.merge_contributions(contributions)

Ignore 'Update cmake presets' (b427d6915c82e20f2c1312b9d647ece6fce3d6ae): presets on backport branch
Ignore 'Write setup Geant4 log messages to world logger, not self logger' (cc6f24d1716c9e2f652f9db3e732cf82f056c76e): backported component from #1895


  0%|          | 0/624 [00:00<?, ?it/s]

In [43]:
make_zenodo_md = release_notes.ZenodoMetadataBuilder(user_cache=user_cache, teams=TEAMS)
zmd = make_zenodo_md(all_contrib, release_md, gh_release)

In [51]:
zmd

{'upload_type': 'software',
 'creators': [{'name': 'Seth R. Johnson',
   'affiliation': 'ORNL',
   'orcid': '0000-0003-1504-8966'},
  {'name': 'Amanda Lund',
   'affiliation': 'ANL',
   'orcid': '0000-0002-8316-0709'},
  {'name': 'Julien Esseiva',
   'affiliation': 'LBNL',
   'orcid': '0009-0002-1119-4851'},
  {'name': 'Elliott Biondo',
   'affiliation': 'ORNL',
   'orcid': '0000-0002-9088-1360'},
  {'name': 'Hayden Hollenbeck',
   'affiliation': 'UVA',
   'orcid': '0009-0000-6359-3294'},
  {'name': 'Rashika Gupta',
   'affiliation': 'ORNL',
   'orcid': '0009-0003-0658-9549'},
  {'name': 'Philippe Canal',
   'affiliation': 'FNAL',
   'orcid': '0000-0002-7748-7887'},
  {'name': 'Stefano Tognini',
   'affiliation': 'ORNL',
   'orcid': '0000-0001-9741-6608'},
  {'name': 'Loren Schwiebert',
   'affiliation': 'Wayne State U',
   'orcid': '0000-0002-7167-7436'},
  {'name': 'Soon Yung Jun',
   'affiliation': 'FNAL',
   'orcid': '0000-0003-3370-6109'},
  {'name': 'Greg Davidson',
   'affiliati

In [53]:
zmd["title"] = title

In [54]:
# Create the Zenodo deposition
deposition = zenodo.create_deposition(zmd)

Created deposition 23325804 at https://zenodo.org/deposit/23325804 : Celeritas 0.7


In [ ]:
zmd

{'upload_type': 'software',
 'creators': [{'name': 'Seth R. Johnson',
   'affiliation': 'ORNL',
   'orcid': '0000-0003-1504-8966'},
  {'name': 'Amanda Lund',
   'affiliation': 'ANL',
   'orcid': '0000-0002-8316-0709'},
  {'name': 'Julien Esseiva',
   'affiliation': 'LBNL',
   'orcid': '0009-0002-1119-4851'},
  {'name': 'Elliott Biondo',
   'affiliation': 'ORNL',
   'orcid': '0000-0002-9088-1360'},
  {'name': 'Hayden Hollenbeck',
   'affiliation': 'UVA',
   'orcid': '0009-0000-6359-3294'},
  {'name': 'Philippe Canal',
   'affiliation': 'FNAL',
   'orcid': '0000-0002-7748-7887'},
  {'name': 'Rashika Gupta',
   'affiliation': 'ORNL',
   'orcid': '0009-0003-0658-9549'},
  {'name': 'Stefano Tognini',
   'affiliation': 'ORNL',
   'orcid': '0000-0001-9741-6608'},
  {'name': 'Loren Schwiebert',
   'affiliation': 'Wayne State U',
   'orcid': '0000-0002-7167-7436'},
  {'name': 'Soon Yung Jun',
   'affiliation': 'FNAL',
   'orcid': '0000-0003-3370-6109'},
  {'name': 'Greg Davidson',
   'affiliati

In [ ]:
# Upload the release tarball
deposition.upload(artifact_tgz, f"celeritas-{release_md.release}.tar.gz")

print(deposition.html)
print("""
IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses
""")

Uploaded celeritas-0.6.0.tar.gz: version bbd63016-30a1-4be9-9a74-329ca975bed9
https://zenodo.org/deposit/15281110

IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses

